In [ ]:
from pureskillgg_makenew_pyskill.notebook import setup_notebook

In [ ]:
setup_notebook()

In [ ]:
# %load ../usual_suspects.py
# pylint: disable=unused-import
import time
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pureskillgg_dsdk.tome import create_tome_curator

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.min_rows", 150)
# pd.set_option('display.float_format', '{:.4f}'.format)

curator = create_tome_curator()

Build a tome per channel with `build_basic_tomes`: every row is tagged `match_key`, and days from different data revisions mix fine. If each match needs summarizing before it's stored (say, `player_vector`), use the `make_tome` loop below instead.

In [ ]:
basic = curator.build_basic_tomes([
    {"channel": "player_info", "columns": ["player_id_fixed", "rank", "rank_type", "wins", "round"]},
    {"channel": "weapon_fire", "columns": ["tick", "weapon_name", "player_id_fixed"]},
])
weapon_fire = basic.tomes["weapon_fire"].get_dataframe()
weapon_fire.head()

In [ ]:
def extract_cool_df(data):
    """Replace this with your own transform: one summary row per player per round."""
    vector = data['player_vector']
    return (
        vector.groupby(['round', 'player_id_fixed'], as_index=False)
        .agg(duck_share=('is_ducked', 'mean'), mean_recoil=('recoil_index', 'mean'))
    )

In [ ]:
tomer = curator.make_tome(
    'cool_tome.2026-10-09,2026-10-10',
    max_page_size_mb=1,
    ds_reading_instructions=[
        {
            "channel": 'player_vector',
            "columns": ['round', 'player_id_fixed', 'is_ducked', 'recoil_index']
        },
    ])

In [ ]:
for data, key in tomer.iterate():
    df = extract_cool_df(data)
    df['match_key'] = key
    tomer.concat(df)

In [ ]:
curator.get_dataframe('cool_tome.2026-10-09,2026-10-10').head()